# 02 — Find the CUDA performance bottleneck with torch.profiler

**Run after the eager benchmark**. Profiling has overhead, so this uses only a few batches and never mixes profiler measurements into latency/throughput results.

In [1]:
from pathlib import Path
import subprocess, sys, json

def find_analysis_folder():
    current = Path.cwd().resolve()
    for parent in [current, *current.parents]:
        for relative in [Path('notebooks/runtime_analysis'), Path('runtime_analysis'), Path('.')]:
            candidate = (parent / relative).resolve()
            if (candidate / 'config.json').is_file() and (candidate / 'worker.py').is_file():
                return candidate
    raise FileNotFoundError('Cannot locate notebooks/runtime_analysis/config.json. Open this notebook inside the copied repository.')

ANALYSIS_DIR = find_analysis_folder()
REPO_ROOT = ANALYSIS_DIR.parent.parent
CONFIG = json.loads((ANALYSIS_DIR/'config.json').read_text())
print('Folder:', ANALYSIS_DIR)
print('Repository:', REPO_ROOT)

Folder: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/notebooks/runtime_analysis
Repository: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers


## Profile seed 42, eager FP32 and INT8 independently

Outputs a `profiler_operators.csv` and Chrome trace for each variant. CSV event times highlight costly operators; open trace JSON in a compatible trace viewer to inspect CUDA kernels.

In [2]:
SEEDS = [42]
cmd = [sys.executable, str(ANALYSIS_DIR/'run_experiments.py'), '--seeds', *map(str, SEEDS), '--variants', 'fp32_eager', 'int8_eager', '--task', 'profile']
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

CUDA variants run SEQUENTIALLY in a FRESH PROCESS each.
Working directory: /home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers

Running seed=42, variant=fp32_eager, task=profile



W1025 22:51:23.471000 45102 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1025 22:51:23.487000 45102 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 18849.75it/s]
USDT:2026-10-25 22:51:24 45102:45102 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-10-25 22:51:24 45102:45102 SyncActivityProfilerHandler.cpp:46] profiler_stop


Profiler summary (device self-time; operator breakdown):
                                                                                                                                                                                                                                                                                                                                                                                                                                                 operator  count  self_cpu_time_us  self_device_time_us  cpu_memory_bytes  device_memory_bytes
                                                                                                                                                                                                                                                                                                                                                                                                                                

W1025 22:51:29.228000 45187 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1025 22:51:29.244000 45187 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15931.05it/s]
USDT:2026-10-25 22:51:31 45187:45187 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-10-25 22:51:32 45187:45187 SyncActivityProfilerHandler.cpp:46] profiler_stop


Profiler summary (device self-time; operator breakdown):
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              operator  count  self_cpu_time_us  self_device_time_us  cpu_memory_bytes  device_memory_bytes
                   

CompletedProcess(args=['/home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/.venv/bin/python', '/home/mohnishr/workspace/DLBA/Quantization-Spurious-Heuristics-in-Transformers/notebooks/runtime_analysis/run_experiments.py', '--seeds', '42', '--variants', 'fp32_eager', 'int8_eager', '--task', 'profile'], returncode=0)

## Compare expensive operators

These are profiler-reported operator self-times, **not end-to-end wall-clock time**. Profiling itself slows execution.

In [3]:
import pandas as pd
for variant in ['fp32_eager', 'int8_eager']:
    p = REPO_ROOT / CONFIG['results_dir'] / 'seed42' / variant / 'profiler_operators.csv'
    df = pd.read_csv(p)
    print(f'\n{variant} — top GPU operator time')
    display(df.sort_values('self_device_time_us', ascending=False).head(20))


fp32_eager — top GPU operator time


,operator,count,self_cpu_time_us,self_device_time_us,cpu_memory_bytes,device_memory_bytes
0,aten::addmm,592,3650.347,60424.897,0,1573654528
1,void cutlass::Kernel2<cutlass_80_simt_sgemm_25...,384,0.000,29772.237,0,0
2,void cutlass::Kernel2<cutlass_80_simt_sgemm_12...,96,0.000,15471.198,0,0
3,"void magma_sgemmEx_kernel<float, float, float,...",96,0.000,14709.954,0,0
4,aten::_efficient_attention_forward,96,596.098,3357.625,0,201424896
5,fmha_cutlassF_f32_aligned_64x64_rf_sm80(PyTorc...,96,0.000,3357.625,0,0
6,aten::native_layer_norm,200,1090.342,728.247,0,420249600
7,void at::native::(anonymous namespace)::vector...,200,0.000,728.247,0,0
8,void at::native::vectorized_elementwise_kernel...,96,0.000,497.516,0,0
9,aten::gelu,96,325.211,497.516,0,566231040



int8_eager — top GPU operator time


,operator,count,self_cpu_time_us,self_device_time_us,cpu_memory_bytes,device_memory_bytes
0,void cutlass::Kernel2<cutlass_80_wmma_tensorop...,584,0.000,19881.735,0,0
1,aten::_int_mm,584,7383.469,19881.735,0,1429209088
2,aten::copy_,8944,18057.620,17183.468,0,0
3,"void at::native::elementwise_kernel<128, 4, at...",584,0.000,11877.859,0,0
4,aten::mul,2368,9531.932,7137.494,0,4197072896
5,aten::_local_scalar_dense,21321,29867.450,6990.734,0,0
6,Memcpy DtoH (Device -> Pinned),21320,0.000,6990.734,0,0
7,aten::cat,4144,13276.330,4195.794,0,2121728
8,void at::native::(anonymous namespace)::CatArr...,4144,0.000,4195.794,0,0
9,"void at::native::elementwise_kernel<128, 2, at...",1184,0.000,3847.850,0,0


## Interpretation

Look for large linear/matmul operations, quantize/dequantize overhead, and unexpectedly high elementwise overhead. Operator tables are aggregated and cannot by themselves prove exact cause—inspect trace timelines.

Next: **`03_compiled_experiment.ipynb`** (optional, potentially resource-intensive).